In [1]:
import os
from typing import List, Dict, Any

In [2]:
from langchain_core.documents import Document


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Create a simple Doc


In [ ]:
doc= Document(
    page_content= "Hello, this is a sample document for testing the data ingestion process in our RAG system. It contains multiple sentences and paragraphs to ensure that the text splitting functionality works correctly.",
    metadata={"source":"example.txt","page":1, "author":"John Doe", "keywords": ["sample", "document", "testing", "data ingestion", "RAG system"], "created_at": "2024-06-01T12:00:00Z"},
)

print("document structured with metadata")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")



document structured with metadata
Content: Hello, this is a sample document for testing the data ingestion process in our RAG system. It contains multiple sentences and paragraphs to ensure that the text splitting functionality works correctly.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'keywords': ['sample', 'document', 'testing', 'data ingestion', 'RAG system'], 'created_at': '2024-06-01T12:00:00Z'}


## Why metadata is important
# => Metadata is crucial for filtering search results, tracking documents sources, providing context in responses, debuggin and auditing 


In [7]:
# other splitters also
from langchain_text_splitters import (
    CharacterTextSplitter,
    TokenTextSplitter
)

## create a simple text file


In [9]:
os.makedirs("data/text_files",exist_ok=True)

In [15]:
sample_text_file = {
    "data/text_files/sample1.txt": """
Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named Arjun Dhurandhar. Known for his strategic mind and relentless determination, Arjun works for a covert national security unit tasked with preventing high-level threats to the country.

The story begins when Arjun uncovers a dangerous international conspiracy involving illegal weapons trade and political manipulation. As he dives deeper into the investigation, he discovers that powerful figures within the system may also be involved. This forces him to operate outside official channels.

Throughout the movie, Arjun Dhurandhar demonstrates exceptional analytical skills, tactical combat abilities, and psychological insight. His character represents discipline, patriotism, and moral conflict as he struggles between following orders and doing what is ethically right.

The narrative combines intense action sequences with strategic intelligence operations. Themes explored in the film include loyalty, corruption, national security, and personal sacrifice.

""",
"data/text_files/sample2.txt": """
Doctor Doom, also known as Victor Von Doom, is one of the most iconic villains in the Marvel universe. He is the ruler of the fictional European country Latveria and is known for his genius-level intellect, mastery of advanced technology, and deep knowledge of mystical arts.

Victor Von Doom was once a brilliant scientist who studied alongside Reed Richards. After a laboratory accident scarred his face, Doom blamed Richards and became obsessed with proving his superiority. This obsession shaped his transformation into the armored supervillain known as Doctor Doom.

Doctor Doom's suit of armor grants him superhuman strength, durability, energy projection abilities, and advanced weapon systems. Beyond technology, Doom has also studied dark magic and mystical powers, making him both a scientific and magical threat.

Despite being a villain, Doom often believes he is the only one capable of ruling the world properly. His motivations are driven by pride, control, and a belief that his leadership would create order and stability.

"""

}

for filePath, content in sample_text_file.items():
    with open(filePath, "w", encoding= "utf-8") as f:
        f.write(content)


print("Sample text file created at: data/text_files/sample1.txt")

Sample text file created at: data/text_files/sample1.txt


## Text loader -> read single file


In [23]:
from langchain_community.document_loaders import TextLoader

#pass entire path of file to loader
loader = TextLoader("data/text_files/sample1.txt",encoding="utf-8" )
document=loader.load()
print(f"type of document: {type(document)}")
print(f"Content: {document[0].page_content}")
print(f"Metadata: {document[0].metadata}")


type of document: <class 'list'>
Content: 
Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named Arjun Dhurandhar. Known for his strategic mind and relentless determination, Arjun works for a covert national security unit tasked with preventing high-level threats to the country.

The story begins when Arjun uncovers a dangerous international conspiracy involving illegal weapons trade and political manipulation. As he dives deeper into the investigation, he discovers that powerful figures within the system may also be involved. This forces him to operate outside official channels.

Throughout the movie, Arjun Dhurandhar demonstrates exceptional analytical skills, tactical combat abilities, and psychological insight. His character represents discipline, patriotism, and moral conflict as he struggles between following orders and doing what is ethically right.

The narrative combines intense action sequences with strategic intelligence operations

## DirectoryLoader -> Multiple text file 

In [24]:
from langchain_community.document_loaders import DirectoryLoader


In [26]:
dirLoader= DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True, ##laoder class to use
    loader_kwargs={"encoding":"utf-8"}
)

dirdocuments=dirLoader.load()
print(f"Number of documents loaded: {len(dirdocuments)}")
for i, doc in enumerate(dirdocuments):
    print(f"Document {i+1}:")
    print(f"Content: {doc.page_content[:100]}...")  # Print first 100 characters
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

100%|██████████| 2/2 [00:00<00:00, 1587.85it/s]

Number of documents loaded: 2
Document 1:
Content: 
Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named A...
Metadata: {'source': 'data\\text_files\\sample1.txt'}
--------------------------------------------------
Document 2:
Content: 
Doctor Doom, also known as Victor Von Doom, is one of the most iconic villains in the Marvel univer...
Metadata: {'source': 'data\\text_files\\sample2.txt'}
--------------------------------------------------


# Text Splitting strategies


## Character text splitter 
### do splitting based on characters


In [34]:

print(document[0].page_content)

print("Character text splitter implementation")

char_splitter= CharacterTextSplitter(
   # separator= "\n", # split on new lines
    chunk_size= 300, # max characters in each chunk,
    chunk_overlap=20, #overlap between chunks to maintain context
    length_function = len

)

char_chunks = char_splitter.split_text(document[0].page_content)
print(f"Number of chunks created: {len(char_chunks)}")
for i , chunk in enumerate(char_chunks):
    print(f"Chunk {i+1}: {chunk[:100]}...") # print first 100 characters of each chunk

Created a chunk of size 305, which is longer than the specified 300



Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named Arjun Dhurandhar. Known for his strategic mind and relentless determination, Arjun works for a covert national security unit tasked with preventing high-level threats to the country.

The story begins when Arjun uncovers a dangerous international conspiracy involving illegal weapons trade and political manipulation. As he dives deeper into the investigation, he discovers that powerful figures within the system may also be involved. This forces him to operate outside official channels.

Throughout the movie, Arjun Dhurandhar demonstrates exceptional analytical skills, tactical combat abilities, and psychological insight. His character represents discipline, patriotism, and moral conflict as he struggles between following orders and doing what is ethically right.

The narrative combines intense action sequences with strategic intelligence operations. Themes explored in the film include loya

## M2:- Recursive Character Splitter
### here we split text data into chunks using recursion , it takes a large text and splits it based on a specified chunk size. It does this by using a set of characters. The default characters provided to it are ["\n\n", "\n", " ", ""].

In [42]:
recur_chunk= RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], # split on paragraphs, then lines, then words, then characters
    chunk_size=100,
    chunk_overlap=20,
    length_function = len
)
recur_chunks = recur_chunk.split_text(document[0].page_content)
print(f"Number of chunks created: {len(recur_chunks)}")
for i , chunk in enumerate(recur_chunks):
    print(f"Chunk {i+1}: {chunk} --") # print first 100 characters of each chunk  

Number of chunks created: 15
Chunk 1: Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named --
Chunk 2: officer named Arjun Dhurandhar. Known for his strategic mind and relentless determination, Arjun --
Chunk 3: Arjun works for a covert national security unit tasked with preventing high-level threats to the --
Chunk 4: threats to the country. --
Chunk 5: The story begins when Arjun uncovers a dangerous international conspiracy involving illegal weapons --
Chunk 6: illegal weapons trade and political manipulation. As he dives deeper into the investigation, he --
Chunk 7: investigation, he discovers that powerful figures within the system may also be involved. This --
Chunk 8: be involved. This forces him to operate outside official channels. --
Chunk 9: Throughout the movie, Arjun Dhurandhar demonstrates exceptional analytical skills, tactical combat --
Chunk 10: tactical combat abilities, and psychological insight. His character represents d

### one thing to note 

 | Feature                   | CharacterTextSplitter     | RecursiveCharacterTextSplitter                           | TokenTextSplitter           | MarkdownHeaderTextSplitter      |
| ------------------------- | ------------------------- | -------------------------------------------------------- | --------------------------- | ------------------------------- |
| **Splitting logic**       | Fixed character length    | Hierarchical separators (paragraph → line → word → char) | Token count using tokenizer | Based on markdown headings      |
| **Primary goal**          | Simple chunk size control | Preserve text structure                                  | Respect LLM token limits    | Preserve document structure     |
| **Unit of splitting**     | Characters                | Characters + separators                                  | Tokens                      | Sections under headers          |
| **Semantic preservation** | Poor                      | Moderate                                                 | Moderate                    | High (for structured docs)      |
| **Chunk size control**    | Character count           | Character count                                          | Token count                 | Section based                   |
| **Separator awareness**   | Only one separator        | Multiple fallback separators                             | None                        | Headers (`#`, `##`, etc.)       |
| **Best use case**         | Quick prototypes          | General RAG pipelines                                    | Token-sensitive models      | Docs, manuals, technical guides |
| **LLM compatibility**     | May break token limits    | May break token limits                                   | Exact token-safe chunks     | Depends on section size         |
| **Speed**                 | Fastest                   | Slightly slower                                          | Medium (tokenization cost)  | Medium                          |
| **Production usage**      | Rare                      | Very common                                              | Common                      | Used for documentation search   |
| **Major weakness**        | Breaks sentences          | Still not semantic-aware                                 | Tokenization overhead       | Only works on markdown          |


## Token based Splitting



In [45]:
token_chunk= TokenTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    encoding_name="gpt2",
)
token_chunks = token_chunk.split_text(document[0].page_content)
print(f"Number of token chunks created: {len(token_chunks)}")
for i , chunk in enumerate(token_chunks):
    print(f"Chunk {i+1}: {chunk}...") # print first 100 characters of each chunk

Number of token chunks created: 3
Chunk 1: 
Dhurandhar is a fictional action-drama film centered around a fearless intelligence officer named Arjun Dhurandhar. Known for his strategic mind and relentless determination, Arjun works for a covert national security unit tasked with preventing high-level threats to the country.

The story begins when Arjun uncovers a dangerous international conspiracy involving illegal weapons trade and political manipulation. As he dives deeper into the investigation, he discovers that powerful figures within the system may also be involved....
Chunk 2:  he dives deeper into the investigation, he discovers that powerful figures within the system may also be involved. This forces him to operate outside official channels.

Throughout the movie, Arjun Dhurandhar demonstrates exceptional analytical skills, tactical combat abilities, and psychological insight. His character represents discipline, patriotism, and moral conflict as he struggles between following 